# Notebook 02 · Image preparation, part 1: thick-slice series

Prerequisite: notebook `01_dados_cq500_bhx.ipynb` already run (CQ500 archives, BHX CSV files, `reads.csv`
and `cq500_mapa_exames.csv` on the Drive). Runs on a CPU; no GPU is needed.

What this part does, in order:

1. **Inventory** of all series of the 491 examinations (one DICOM header per series, read directly from
   the archive).
2. **Selection of the thick-slice series**: in examinations with boxes, the series annotated by BHX; in the
   others, a general rule, validated against the BHX series (rule 2, 24 Oct 2026).
3. **Reading in HU, three windows and boxes** for one examination, with a check figure.
4. **Counts** of thick slices with and without haemorrhage per group of patients.

Decision: train on thick slices, like those of CT-ICH (5 mm), using the hand-drawn BHX boxes
(`labelType = handDrawn`: 6,283 boxes on 3,558 images of 233 series). The first run in Colab
(24 Oct 2026) showed that 193 of these series have 5 mm slices, 30 have 3 mm slices and 10 are thin-slice.

Outside Colab (local test), set the environment variable `PREPARO_LOCAL` to a folder that mimics `dados/`
on the Drive (`tc/bhx/`, `tc/cq500/zips/`, `tc/cq500/reads.csv`, `tc/cq500/cq500_mapa_exames.csv`).

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Folders and libraries. In Colab the Drive is used; elsewhere, the folder given in PREPARO_LOCAL.
import ast, csv, io, json, os, re, subprocess, sys, zipfile, collections
import concurrent.futures as cf
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DADOS = Path('/content/drive/MyDrive/ich-benchmark/dados')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pydicom', 'pylibjpeg', 'pylibjpeg-libjpeg'], check=True)
except ImportError:
    DADOS = Path(os.environ['PREPARO_LOCAL'])

import numpy as np
import pydicom   # with pylibjpeg: some CQ500 examinations are stored as lossless JPEG

BHX, CQ500 = DADOS / 'tc' / 'bhx', DADOS / 'tc' / 'cq500'
PREPARO = DADOS / 'tc' / 'preparo'          # outputs of this step
PREPARO.mkdir(parents=True, exist_ok=True)
print(BHX, CQ500, PREPARO, sep='\n')

In [ ]:
# 2. Hand-drawn BHX boxes (clean version) and CQ500 examinations
bhx = [l for l in csv.DictReader(open(BHX / '3_Extrapolation_to_Selected_Series.csv'))
       if l['labelType'] == 'handDrawn']
mapa = list(csv.DictReader(open(CQ500 / 'cq500_mapa_exames.csv')))
uid_para_nome = {m['StudyInstanceUID']: m['Name'] for m in mapa if m['StudyInstanceUID']}

series_bhx = collections.defaultdict(set)        # examination -> series with hand-drawn boxes
caixas_por_corte = collections.defaultdict(list)  # SOPInstanceUID -> [(subtype, x, y, width, height)]
for l in bhx:
    d = ast.literal_eval(l['data'])               # the CSV stores the dictionary as text: {'x':..., 'y':..., 'width':..., 'height':...}
    series_bhx[uid_para_nome[l['StudyInstanceUID']]].add(l['SeriesInstanceUID'])
    caixas_por_corte[l['SOPInstanceUID']].append((l['labelName'], d['x'], d['y'], d['width'], d['height']))
print(len(bhx), 'caixas à mão;', len(caixas_por_corte), 'cortes;',
      sum(len(s) for s in series_bhx.values()), 'séries;', len(series_bhx), 'exames')

In [ ]:
# 3. Inventory: one row per series of each examination (reads only the first header of each series, from the archive)
def zip_do_exame(nome):
    recuperado = CQ500 / 'zips_recuperados' / f'{nome}.zip'
    return recuperado if recuperado.exists() else CQ500 / 'zips' / f'{nome}.zip'

def numero(valor, padrao=float('nan')):
    try:
        return float(valor)
    except (TypeError, ValueError):
        return padrao

def series_do_exame(nome):
    caminho = zip_do_exame(nome)
    if not caminho.exists():
        return [{'Name': nome, 'erro': 'zip ausente'}]
    linhas = []
    try:
        with zipfile.ZipFile(caminho) as z:
            por_pasta = collections.defaultdict(list)
            for m in z.namelist():
                if m.lower().endswith('.dcm'):
                    por_pasta[m.rsplit('/', 2)[-2]].append(m)
            for pasta, arquivos in sorted(por_pasta.items()):
                c = pydicom.dcmread(io.BytesIO(z.read(sorted(arquivos)[0])), stop_before_pixels=True)
                px = c.get('PixelSpacing', [None, None])
                linhas.append({
                    'Name': nome, 'pasta': pasta, 'cortes': len(arquivos),
                    'SeriesInstanceUID': c.get('SeriesInstanceUID', ''),
                    'espessura_mm': numero(c.get('SliceThickness')),
                    'kernel': str(c.get('ConvolutionKernel', '')).upper(),
                    'descricao': str(c.get('SeriesDescription', '')),
                    'tipo_imagem': '/'.join(c.get('ImageType', [])),
                    'pixel_mm': numero(px[0]), 'linhas': c.get('Rows'), 'colunas': c.get('Columns'),
                    'contraste': str(c.get('ContrastBolusAgent', '')),
                    'caixas_a_mao': c.get('SeriesInstanceUID', '') in series_bhx.get(nome, set()),
                    'erro': ''})
    except Exception as erro:
        linhas.append({'Name': nome, 'erro': f'{type(erro).__name__}: {erro}'})
    return linhas

with cf.ThreadPoolExecutor(8) as ex:
    series = [s for lista in ex.map(series_do_exame, [m['Name'] for m in mapa]) for s in lista]
campos = ['Name', 'pasta', 'cortes', 'SeriesInstanceUID', 'espessura_mm', 'kernel', 'descricao', 'tipo_imagem',
          'pixel_mm', 'linhas', 'colunas', 'contraste', 'caixas_a_mao', 'erro']
with open(PREPARO / 'cq500_series.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=campos); w.writeheader(); w.writerows(series)
validas = [s for s in series if not s['erro']]
print(len(validas), 'séries em', len({s['Name'] for s in validas}), 'exames;',
      sum(s['caixas_a_mao'] for s in validas), 'séries com caixa à mão')
print('Com erro:', collections.Counter(s['erro'] for s in series if s['erro']))

# How the series annotated by BHX compare with the others (basis for the rule of cell 4)
def resumo(grupo):
    return {'séries': len(grupo),
            'espessura': collections.Counter(s['espessura_mm'] for s in grupo).most_common(6),
            'kernel': collections.Counter(s['kernel'] for s in grupo).most_common(8),
            'pixel_mm (mediana)': float(np.median([s['pixel_mm'] for s in grupo])) if grupo else None}
print('Marcadas pela BHX:', json.dumps(resumo([s for s in validas if s['caixas_a_mao']]), ensure_ascii=False))
print('Outras:          ', json.dumps(resumo([s for s in validas if not s['caixas_a_mao']]), ensure_ascii=False))

In [ ]:
# 4. Selection of the thick-slice series of each examination (rule 2, 24 Oct 2026)
#    Examinations with hand-drawn boxes: the series annotated by BHX, because the boxes are tied to its slices.
#    Other examinations: general rule. For validation, the general rule is also applied to the examinations with boxes and
#    counts as a match if it picks the annotated series or a copy of it (same thickness, kernel, pixel size and number of slices).
GROSSA_MIN, GROSSA_MAX = 2.5, 6.0                  # mm; includes the 3 mm slices (30 series annotated by BHX)
FILTROS_DUROS = ('BONE', 'LUNG', 'SHARP')           # contained in the kernel name
FILTROS_DUROS_EXATOS = {'EC', 'DETAIL'}
# CQ500 headers do not record contrast (empty field in all 1,274 series): the series name is used.
# "PRE CONTRAST" means before contrast, hence non-contrast; "C", "CECT", "POST", "150cc", "ORAL IV" indicate contrast.
NOME_CONTRASTE = re.compile(r'POST|(?<!PRE )CONTRAST|(^|[ _-])C([ _-]|$)|CECT|\d+ ?CC\b|ORAL|\bIV\b', re.I)
NOME_DURO = re.compile(r'BONE|DETAIL', re.I)     # bone series by name, even with an unknown kernel

def grossa(s):
    return GROSSA_MIN <= s['espessura_mm'] <= GROSSA_MAX

def candidata(s):
    return (grossa(s) and s['cortes'] >= 20 and not s['contraste']
            and not NOME_CONTRASTE.search(f"{s['pasta']} {s['descricao']}")
            and not NOME_DURO.search(f"{s['pasta']} {s['descricao']}")
            and not any(f in s['kernel'] for f in FILTROS_DUROS) and s['kernel'] not in FILTROS_DUROS_EXATOS)

def chave(s):
    plain = 'PLAIN' in f"{s['pasta']} {s['descricao']}".upper()
    return (abs(s['espessura_mm'] - 5), not plain, round(s['pixel_mm'], 2), -s['cortes'], s['pasta'])

def copia(a, b):
    return (a['espessura_mm'] == b['espessura_mm'] and a['kernel'] == b['kernel']
            and abs(a['pixel_mm'] - b['pixel_mm']) < 0.01 and a['cortes'] == b['cortes'])

por_exame = collections.defaultdict(list)
for s in validas:
    por_exame[s['Name']].append(s)
regra = {n: (sorted(filter(candidata, l), key=chave) or [None])[0] for n, l in por_exame.items()}
marcadas = {n: [s for s in l if s['caixas_a_mao']] for n, l in por_exame.items() if series_bhx.get(n)}
positivos = sorted(marcadas)

exata = [n for n in positivos if regra[n] and regra[n]['caixas_a_mao']]
por_copia = [n for n in positivos if regra[n] and not regra[n]['caixas_a_mao']
             and any(copia(regra[n], m) for m in marcadas[n])]
acertos = exata + por_copia
diverge = sorted(set(positivos) - set(acertos))
print(f'Regra geral × BHX nos {len(positivos)} exames com caixa: {len(exata)} na série marcada, '
      f'{len(por_copia)} numa cópia dela, {len(diverge)} divergentes')
for n in diverge:
    r = regra[n]
    print(' ', n, '| regra:', r and (r['pasta'], r['espessura_mm'], r['kernel'], r['pixel_mm'], r['cortes']),
          '| BHX:', [(s['pasta'], s['espessura_mm'], s['kernel'], s['pixel_mm'], s['cortes']) for s in marcadas[n]])

# Series used: the one annotated by BHX in examinations with boxes (the closest to 5 mm and, among those, the non-bone
# one, if there is more than one); the one chosen by the rule in the others
escolhida = {n: (sorted(marcadas[n], key=lambda s: (abs(s['espessura_mm'] - 5),
                                                    bool(NOME_DURO.search(s['pasta'])), s['pasta']))[0]
                 if n in marcadas else regra[n]) for n in por_exame}
so_finos = sorted(n for n in positivos if not grossa(escolhida[n]))
com_contraste = sorted(n for n in positivos if NOME_CONTRASTE.search(escolhida[n]['pasta']))
print('Exames com caixa só em cortes finos (fora da análise principal):', so_finos)
print('Exames com caixa só em série com contraste (fora da análise principal):', com_contraste)
print('Exames sem série grossa:', sorted(n for n, s in escolhida.items() if s is None))
print('Espessura das séries usadas:', collections.Counter(s['espessura_mm'] for s in escolhida.values() if s))

In [ ]:
# 5. Reading a series in HU, three windows as three channels, and the BHX boxes
JANELAS = {'cérebro': (40, 80), 'subdural': (80, 200), 'osso': (600, 2800)}   # (level, width) in HU

def ler_serie(nome, pasta):
    """Volume em HU (cortes ordenados de baixo para cima), SOPInstanceUID de cada corte e espaçamento."""
    cortes = []
    with zipfile.ZipFile(zip_do_exame(nome)) as z:
        for m in z.namelist():
            if m.lower().endswith('.dcm') and m.rsplit('/', 2)[-2] == pasta:
                d = pydicom.dcmread(io.BytesIO(z.read(m)))
                hu = d.pixel_array.astype(np.float32) * float(d.RescaleSlope) + float(d.RescaleIntercept)
                cortes.append((float(d.ImagePositionPatient[2]), d.SOPInstanceUID, hu, d))
    cortes.sort(key=lambda c: c[0])
    d = cortes[0][3]
    return (np.stack([c[2] for c in cortes]), [c[1] for c in cortes],
            (float(d.PixelSpacing[0]), float(d.PixelSpacing[1]), float(d.SliceThickness)))

def janelar(hu, nivel, largura):
    return np.clip((hu - (nivel - largura / 2)) / largura, 0, 1)

def tres_canais(hu):
    """Imagem de 3 canais (cérebro, subdural, osso), cada um de 0 a 1 — a entrada comum dos dois detectores."""
    return np.stack([janelar(hu, *JANELAS[j]) for j in ('cérebro', 'subdural', 'osso')], axis=-1)

In [ ]:
# 6. Check figure: one examination with haemorrhage, the three windows and the BHX boxes overlaid
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

EXEMPLO = 'CQ500-CT-173'           # 32 thick 5 mm slices, 52 hand-drawn boxes
serie = escolhida[EXEMPLO]
vol, sops, espac = ler_serie(EXEMPLO, serie['pasta'])
k = max(range(len(sops)), key=lambda i: len(caixas_por_corte.get(sops[i], [])))   # slice with the most boxes
img = tres_canais(vol[k])
cores = {'Intraparenchymal': '#e11d48', 'Intraventricular': '#2563eb', 'Subarachnoid': '#f59e0b',
         'Subdural': '#16a34a', 'Chronic': '#7c3aed', 'Epidural': '#0891b2'}
fig, eixos = plt.subplots(1, 4, figsize=(16, 4.6))
# Figure text in English: the figure is used in the article
for ax, (titulo, im) in zip(eixos, [(f'Brain window {JANELAS["cérebro"]} HU', img[..., 0]),
                                    (f'Subdural window {JANELAS["subdural"]} HU', img[..., 1]),
                                    (f'Bone window {JANELAS["osso"]} HU', img[..., 2]),
                                    ('3-channel input + BHX boxes', img)]):
    ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
    ax.set_title(titulo, fontsize=10); ax.axis('off')
for tipo, x, y, w, h in caixas_por_corte.get(sops[k], []):
    eixos[3].add_patch(Rectangle((x, y), w, h, fill=False, lw=1.5, color=cores.get(tipo, 'white')))
    eixos[3].text(x, y - 4, tipo, color=cores.get(tipo, 'white'), fontsize=7)
fig.suptitle(f'{EXEMPLO} · series "{serie["pasta"]}" · slice {k + 1} of {len(sops)} · '
             f'pixel {espac[0]:.2f} mm, slice thickness {espac[2]:.0f} mm · CQ500 (CC BY-NC-SA 4.0) + BHX', fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):                     # PNG for the screen, PDF for LaTeX
    fig.savefig(PREPARO / f'f3-04-exemplo-janelas.{ext}', dpi=150)
print('Faixa de HU do volume:', float(vol.min()), 'a', float(vol.max()), '| caixas no corte:', len(caixas_por_corte.get(sops[k], [])))

In [ ]:
# 7. Thick-slice counts per group of patients (from the inventory only, without opening the images)
leituras = {l['name']: l for l in csv.DictReader(open(CQ500 / 'reads.csv'))}
votos = lambda n: sum(int(leituras[n][f'R{r}:ICH']) for r in (1, 2, 3))
sop_da_serie = collections.defaultdict(set)
for l in bhx:
    sop_da_serie[l['SeriesInstanceUID']].add(l['SOPInstanceUID'])

def grupo(nome):
    s = escolhida.get(nome)
    if nome in marcadas:
        if not grossa(s):
            return 'com sangramento, só cortes finos'
        return 'com sangramento, série com contraste' if nome in com_contraste else 'com sangramento'
    if s is None:
        return 'sem série grossa'
    return 'normal' if votos(nome) == 0 else 'duvidoso'

contagem = collections.defaultdict(lambda: {'exames': 0, 'cortes': 0, 'cortes_com_caixa': 0})
linhas = []
for nome in sorted(por_exame, key=lambda n: int(n.rsplit('-', 1)[1])):
    s, g = escolhida[nome], grupo(nome)
    cortes = s['cortes'] if s else 0
    com_caixa = len(sop_da_serie.get(s['SeriesInstanceUID'], ())) if s else 0
    contagem[g]['exames'] += 1; contagem[g]['cortes'] += cortes; contagem[g]['cortes_com_caixa'] += com_caixa
    linhas.append({'Name': nome, 'grupo': g, 'votos_ich': votos(nome), 'pasta': s and s['pasta'],
                   'SeriesInstanceUID': s and s['SeriesInstanceUID'], 'espessura_mm': s and s['espessura_mm'],
                   'kernel': s and s['kernel'], 'pixel_mm': s and s['pixel_mm'], 'cortes': cortes,
                   'cortes_com_caixa': com_caixa, 'serie_da_bhx': bool(s and s['caixas_a_mao'])})
with open(PREPARO / 'cq500_series_escolhidas.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=linhas[0].keys()); w.writeheader(); w.writerows(linhas)
resultado = {'regra': 2, 'validacao_regra_geral': {'exames_com_caixa': len(positivos), 'serie_marcada': len(exata),
             'copia_da_marcada': len(por_copia), 'divergentes': diverge},
             'so_cortes_finos': so_finos, 'serie_com_contraste': com_contraste, 'grupos': contagem}
(PREPARO / 'cq500_contagens.json').write_text(json.dumps(resultado, indent=2, ensure_ascii=False))
print(json.dumps(resultado, indent=2, ensure_ascii=False))

## After running

1. Check cell 4: the general rule should select the BHX series (or an identical copy of it) in almost all
   226 examinations with boxes; the listed disagreements show where it differs.
2. The small files written to `tc/preparo/` on the Drive (`cq500_series.csv`, `cq500_series_escolhidas.csv`
   and `cq500_contagens.json`) describe the cohort and are used by notebook 03.

The next notebooks split the patients into five folds, convert the CT-ICH masks into boxes and export the
data to the MONAI and YOLO formats.